In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from rdkit import Chem
from rdkit.Chem import Descriptors

In [2]:
# Load your dataset (assuming your dataset is in a CSV file)
df = pd.read_csv('Pan-cancer_DATASET.csv', encoding="latin1")

In [3]:
# Handle missing values
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)
print(df.head())

C:\Users\Abdul Rahman\AppData\Local\Temp\ipykernel_9424\1943537772.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\Abdul Rahman\AppData\Local\Temp\ipykernel_9424\1943537772.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)


        Drugs  Cell line TCGA classification Tissue  \
0  Belinostat      JVM-3                 CLL  blood   
1  Belinostat      KU812                LCML  blood   
2  Belinostat  SU-DHL-16                DLBC  blood   
3  Belinostat         SR        UNCLASSIFIED  blood   
4  Belinostat  SU-DHL-10                DLBC  blood   

             Tissue sub-type      IC50       AUC  \
0    lymphoid_neoplasm_other  0.012240  0.903809   
1  chronic_myeloid_leukaemia  0.015511  0.027365   
2            B_cell_lymphoma  0.029505  0.010242   
3    lymphoid_neoplasm_other  0.030923  0.043889   
4            B_cell_lymphoma  0.031633  0.006706   

                                     SMILES         model_name gene_symbol  \
0  O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  HCM-SANG-0282-C18    ANKRD33B   
1  O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  HCM-SANG-0527-C18       AXIN1   
2  O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  HCM-SANG-0286-C20        VAPB   
3  O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO 

In [4]:
# Extract valid SMILES list for similarity matching
valid_smiles_dataset = df['SMILES'].dropna().unique().tolist()


In [5]:
# ------------------- Descriptor & Fingerprint Functions -------------------
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'MolWt': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'ExactMolWt': Descriptors.ExactMolWt(mol),
    }

def calculate_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
    return fp.ToBitString()


In [7]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib
import gradio as gr
# --------------------- Apply Descriptors & Fingerprints ---------------------
df['Descriptors'] = df['SMILES'].apply(calculate_descriptors)
df['Fingerprint'] = df['SMILES'].apply(calculate_fingerprint)
df_descriptors = df['Descriptors'].apply(pd.Series)
df = pd.concat([df, df_descriptors], axis=1)
df = df.loc[:, ~df.columns.duplicated()]


[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc2'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc3'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc4'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc5'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc6'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc7'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc8'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc9'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc10'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc11'
[11:54:38] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc12'
[11:54:38] SMILES 

In [8]:
# Final DataFrame: Show the original columns plus the descriptors and fingerprint
print(df[['IC50', 'SMILES', 'AUC'] + list(df_descriptors.columns) + ['Fingerprint']])

           IC50                                             SMILES       AUC  \
0      0.012240           O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  0.903809   
1      0.015511           O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  0.027365   
2      0.029505           O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  0.010242   
3      0.030923           O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  0.043889   
4      0.031633           O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO  0.006706   
...         ...                                                ...       ...   
1668   0.019782  C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...  0.632044   
1669   5.522189  C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...  0.908488   
1670  36.768629  C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...  0.946342   
1671   0.134950  C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...  0.385767   
1672   0.235826  C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...  0.460426   

        MolWt     LogP    TPSA  NumHDon

In [9]:
# -------------------------- Classification Labels --------------------------
df['Activity_Label'] = df['IC50'].apply(lambda x: 'Active' if x < 5 else 'Inactive')


In [10]:
# ------------------------ Feature and Target Setup ------------------------
descriptor_cols = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'ExactMolWt']
X = df[descriptor_cols]
y_activity = df['Activity_Label']
y_drugname = df['Drugs']
y_tissue = df['Tissue']


In [11]:
# ---------------------- Train-test split & Scaling ----------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_activity, test_size=0.2, stratify=y_activity, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'trained_scaler.pkl')

['trained_scaler.pkl']

In [12]:
# ---------------------- Train Random Forest Models ----------------------
model_activity = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
model_drug = RandomForestClassifier().fit(X, y_drugname)
model_tissue = RandomForestClassifier().fit(X, y_tissue)


In [13]:
# Save models
joblib.dump(model_activity, 'trained_model_activity.pkl')
joblib.dump(model_drug, 'trained_model_drug.pkl')
joblib.dump(model_tissue, 'trained_model_tissue.pkl')

['trained_model_tissue.pkl']

In [14]:
# Evaluate
y_pred = model_activity.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

      Active       0.91      1.00      0.95       305
    Inactive       0.00      0.00      0.00        30

    accuracy                           0.91       335
   macro avg       0.46      0.50      0.48       335
weighted avg       0.83      0.91      0.87       335



C:\Users\Abdul Rahman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Abdul Rahman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Abdul Rahman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib
import gradio as gr

# Load dataset
df = pd.read_csv('Pan-cancer_DATASET.csv', encoding="latin1")
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)

# Calculate descriptors
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'MolWt': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'ExactMolWt': Descriptors.ExactMolWt(mol),
    }

# Calculate fingerprint
def calculate_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
    return fp

# Apply calculations
df['Descriptors'] = df['SMILES'].apply(calculate_descriptors)
df = df[df['Descriptors'].notnull()].copy()
df['Fingerprint'] = df['SMILES'].apply(calculate_fingerprint)
df_descriptors = df['Descriptors'].apply(pd.Series)
df = pd.concat([df, df_descriptors], axis=1)
df = df.loc[:, ~df.columns.duplicated()]

# Label creation
df['Activity_Label'] = df['IC50'].apply(lambda x: 'Active' if x < 5 else 'Inactive')

# Feature & label selection
descriptor_cols = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'ExactMolWt']
X = df[descriptor_cols]
y_activity = df['Activity_Label']
y_drugname = df['Drugs']
y_tissue = df['Tissue']

# Train models
model_activity = RandomForestClassifier().fit(X, y_activity)
model_drug = RandomForestClassifier().fit(X, y_drugname)
model_tissue = RandomForestClassifier().fit(X, y_tissue)
# Save models
joblib.dump(model_activity, 'trained_model_activity.pkl')
joblib.dump(model_drug, 'trained_model_drug.pkl')
joblib.dump(model_tissue, 'trained_model_tissue.pkl')
# Evaluate
y_pred = model_activity.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


# Save fingerprints for similarity use
df['Fingerprint'] = df['SMILES'].apply(lambda smi: AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), 2, 1024))

# Prediction Function with Similarity Fallback
def predict_all_outputs(smiles_input):
    smiles_input = smiles_input.strip()
    mol = Chem.MolFromSmiles(smiles_input)

    if mol is None:
        try:
            input_mol = Chem.MolFromSmiles(smiles_input, sanitize=False)
            Chem.SanitizeMol(input_mol)
            input_fp = AllChem.GetMorganFingerprintAsBitVect(input_mol, 2, 1024)
        except:
            input_mol = Chem.MolFromSmiles("C")
            input_fp = AllChem.GetMorganFingerprintAsBitVect(input_mol, 2, 1024)

        similarities = [
            DataStructs.TanimotoSimilarity(input_fp, fp)
            for fp in df['Fingerprint']
        ]
        most_similar_idx = similarities.index(max(similarities))
        descriptors = df.iloc[most_similar_idx][descriptor_cols]
    else:
        descriptors = {
            'MolWt': Descriptors.MolWt(mol),
            'LogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'ExactMolWt': Descriptors.ExactMolWt(mol)
        }

    X_input = pd.DataFrame([descriptors])

    activity_pred = model_activity.predict(X_input)[0]
    prob_pred = model_activity.predict_proba(X_input)[0]
    label_index = model_activity.classes_.tolist().index(activity_pred)
    activity_with_prob = f"{activity_pred} ({prob_pred[label_index]:.2f})"

    drug_name = model_drug.predict(X_input)[0]
    tissue_type = model_tissue.predict(X_input)[0]

    return activity_with_prob, drug_name, tissue_type

# Gradio Interface
iface = gr.Interface(
    fn=predict_all_outputs,
    inputs=gr.Textbox(label="Enter SMILES"),
    outputs=[
        gr.Textbox(label="Predicted Activity"),
        gr.Textbox(label="Predicted Drug Name"),
        gr.Textbox(label="Predicted Tissue Type")
    ],
    title="Universal Drug & Cancer Prediction",
    description="Enter any SMILES string, even slightly invalid ones, and get predictions based on the most similar valid compound."
)

iface.launch()


C:\Users\Abdul Rahman\AppData\Local\Temp\ipykernel_9424\1268424156.py:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\Abdul Rahman\AppData\Local\Temp\ipykernel_9424\1268424156.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
[12:28:23] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc2'
[12:28:23] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc3'
[12:28:23] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc4'
[12:28:23] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc5'
[12:28:23] SMILES Parse Error: unclosed ring for input: 'CCCCCCCCc1ccc(CCC(N)(CO)CO)cc6'
[12:28:23] SMILES Parse Error: unclosed ring for in

              precision    recall  f1-score   support

      Active       0.91      1.00      0.95       305
    Inactive       0.00      0.00      0.00        30

    accuracy                           0.91       335
   macro avg       0.46      0.50      0.48       335
weighted avg       0.83      0.91      0.87       335



[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerator
[12:28:33] DEPRECATION WARNING: please use MorganGenerat

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
